# RAG：检索增强生成完全指南

RAG（Retrieval-Augmented Generation）是解决 LLM 两大固有缺陷的关键技术：
**知识截止日期**和**幻觉问题**。2023 年以来，RAG 已成为 LLM 落地应用中最常见的架构之一。

本章覆盖：
- LLM 为什么会幻觉？RAG 如何从根本上解决
- RAG 完整流水线：文档切块 -> 向量化 -> 存储 -> 检索 -> 生成
- 四种切块策略对比（固定、句子、递归、语义）
- 关键词检索 BM25 vs 语义检索对比
- 混合检索：稀疏 + 稠密向量
- 简单重排序（Reranker）
- RAG 评估指标：忠实性、相关性、完整性
- 常见失败模式与改进方向

**依赖：**
```bash
pip install sentence-transformers chromadb anthropic python-dotenv
```
向量化使用本地模型，无需 GPU。生成部分需要 `ANTHROPIC_API_KEY`。

## 第一章：LLM 的两大固有缺陷

### 缺陷一：知识截止日期（Knowledge Cutoff）

LLM 的知识来自训练数据，训练完成后就「冻结」了。

```
GPT-4 训练截止：2023年4月
Claude 3.5 Sonnet 训练截止：2024年4月

你问它："2024年11月美国大选结果如何？"
-> 它不知道，可能会瞎说
```

对于需要最新信息的场景（新闻、实时数据、最新政策），这是致命问题。

### 缺陷二：幻觉（Hallucination）

LLM 本质是「预测下一个 token 的概率」，不是「检索事实」。
当模型不确定时，它会生成「听起来合理」的内容，而不是承认不知道。

**幻觉的危害：**
- 医疗/法律领域：错误信息可能造成严重后果
- 企业应用：引用不存在的文件、数据
- 教育场景：学生可能把错误信息当作事实

### RAG 的解决思路

不改变模型，而是改变**输入**：

```
传统方式（纯 LLM）：
  问题 -> LLM -> 答案  （LLM 凭记忆回答，可能幻觉）

RAG 方式：
  问题 -> 检索相关文档 -> [问题 + 相关文档] -> LLM -> 基于文档的答案
```

模型的任务从「凭记忆回答」变成「阅读给定文档并回答」，幻觉大幅减少。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["font.sans-serif"] = ["Arial Unicode MS", "SimHei", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
import os, re, math, warnings
from collections import Counter
warnings.filterwarnings("ignore")

# ============================================================
# 直观演示：LLM 幻觉 vs RAG 有参考的回答
# ============================================================
from dotenv import load_dotenv
import anthropic
load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

def ask(question, context=None, model="claude-haiku-4-5-20251001", max_tokens=400):
    if context:
        prompt = (
            f"请根据以下参考文档回答问题。如果文档中没有答案，请明确说明。\n\n"
            f"参考文档：\n{context}\n\n"
            f"问题：{question}"
        )
    else:
        prompt = question
    return client.messages.create(
        model=model, max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}]
    ).content[0].text

# 公司内部文档（LLM 训练时没有）
internal_doc = """
【公司内部文档 - 2024年Q4绩效考核规定】

考核周期：2024年10月1日至2024年12月31日
考核维度：
  A. 工作完成质量（40%）
  B. 团队协作（30%）
  C. 创新贡献（30%）

评分等级：
  S级（90-100分）：季度奖金150%
  A级（80-89分）：季度奖金120%
  B级（70-79分）：季度奖金100%
  C级（60-69分）：季度奖金80%

申诉期：考核结果公布后5个工作日内可提出申诉
"""

question = "我们公司Q4绩效考核中，S级对应的奖金比例是多少？申诉期有多长？"

print("问题:", question)
print()
print("=== 纯 LLM（无文档，可能幻觉）===")
r_no_rag = ask(question)
print(r_no_rag[:300])

print()
print("=== RAG（提供内部文档）===")
r_rag = ask(question, context=internal_doc)
print(r_rag[:300])


## 第二章：RAG 完整流水线

RAG 分为两个阶段：

```
【索引阶段】（离线，一次性运行）

原始文档（PDF/Word/网页）
    ↓  文本提取
原始文本
    ↓  切块（Chunking）
文本块列表 [chunk1, chunk2, chunk3, ...]
    ↓  向量化（Embedding）
向量列表 [(vec1, chunk1), (vec2, chunk2), ...]
    ↓  存入向量数据库
ChromaDB / FAISS / Pinecone / Weaviate

【查询阶段】（在线，每次用户提问时运行）

用户问题
    ↓  向量化
问题向量
    ↓  相似度搜索
Top-K 相关 chunk
    ↓  组装 Prompt
问题 + 相关 chunk -> LLM
    ↓  生成
最终答案（基于检索到的文档）
```

### 核心组件说明

| 组件 | 作用 | 常见选择 |
|------|------|---------|
| 文本切块器 | 把长文档切成适当大小的片段 | 固定长度、句子、递归、语义 |
| 嵌入模型 | 把文本转成向量（用于相似度比较） | text-embedding-ada-002, bge, e5 |
| 向量数据库 | 存储和快速检索向量 | ChromaDB, FAISS, Pinecone |
| LLM | 基于检索结果生成答案 | Claude, GPT-4, Llama |

## 第三章：文档切块（Chunking）策略

切块是 RAG 中最被低估、但影响极大的环节。
**切块太大**：向量难以精确匹配，检索噪音多。
**切块太小**：缺乏上下文，LLM 无法给出完整答案。

我们来实现并对比 4 种常见策略。

In [ ]:
# ============================================================
# 准备测试文档
# ============================================================
sample_doc = """
人工智能（Artificial Intelligence，AI）是计算机科学的一个分支，致力于创建能够模拟人类智能的系统。
自1956年达特茅斯会议以来，AI领域经历了多次起伏，如今正处于历史上最快速的发展阶段。

机器学习是AI的核心子领域，通过让计算机从数据中自动学习规律。
监督学习、无监督学习和强化学习是机器学习的三大主要范式。
深度学习是机器学习的一个子集，利用多层神经网络处理复杂的模式识别任务。

自然语言处理（NLP）是AI的另一个重要分支，专注于让计算机理解和生成人类语言。
大型语言模型（LLM）如GPT系列和Claude，代表了NLP领域的最新突破。
这些模型基于Transformer架构，通过在海量文本上预训练来获得语言理解能力。

计算机视觉（CV）使计算机能够理解和分析图像与视频内容。
目标检测、图像分类、语义分割是CV的核心任务。
卷积神经网络（CNN）和Vision Transformer（ViT）是主流的CV模型架构。

AI在各行业的应用日益广泛：医疗影像诊断、自动驾驶、金融风控、内容推荐等。
与此同时，AI伦理、数据隐私、算法偏见等问题也受到越来越多的关注。
负责任的AI开发需要在技术创新与社会影响之间寻求平衡。
"""

print(f"测试文档：{len(sample_doc)} 个字符，{len(sample_doc.split())} 个词")
print(f"\n文档预览:\n{sample_doc[:200]}...")


In [ ]:
# ============================================================
# 策略1：固定长度切块（Fixed-size Chunking）
# ============================================================
def fixed_size_chunk(text, chunk_size=150, overlap=30):
    chunks = []
    start  = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end].strip())
        start += chunk_size - overlap
    return [c for c in chunks if c]

# 策略2：句子切块（Sentence-based Chunking）
def sentence_chunk(text, sentences_per_chunk=2, overlap=1):
    # 按句号、问号、感叹号切分
    sents   = re.split(r"(?<=[。！？.!?])\s*", text.strip())
    sents   = [s.strip() for s in sents if s.strip()]
    chunks  = []
    for i in range(0, len(sents), sentences_per_chunk - overlap):
        chunk = "".join(sents[i:i + sentences_per_chunk])
        if chunk:
            chunks.append(chunk)
    return chunks

# 策略3：段落切块（Paragraph-based Chunking）
def paragraph_chunk(text, max_chars=300, overlap_chars=50):
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks = []
    current = ""
    for para in paragraphs:
        if len(current) + len(para) <= max_chars:
            current += para + "\n"
        else:
            if current:
                chunks.append(current.strip())
            current = para + "\n"
    if current:
        chunks.append(current.strip())
    return chunks

# 策略4：递归切块（Recursive Chunking，LangChain 默认策略）
def recursive_chunk(text, max_chars=200, overlap=30, separators=None):
    if separators is None:
        separators = ["\n\n", "\n", "。", "，", " ", ""]
    if len(text) <= max_chars:
        return [text]
    for sep in separators:
        if sep in text:
            parts  = text.split(sep)
            chunks = []
            current = ""
            for part in parts:
                piece = part + sep if sep else part
                if len(current) + len(piece) <= max_chars:
                    current += piece
                else:
                    if current:
                        chunks.append(current.strip())
                    # overlap：新块从上一块末尾 overlap 个字符开始
                    current = current[-overlap:] + piece if overlap else piece
            if current:
                chunks.append(current.strip())
            if chunks:
                return [c for c in chunks if c]
    return [text]  # 最小切块单元

# 对比四种策略
strategies = [
    ("固定长度 (150字/30重叠)", fixed_size_chunk(sample_doc, 150, 30)),
    ("句子切块 (2句/1重叠)",    sentence_chunk(sample_doc, 2, 1)),
    ("段落切块 (max=300)",      paragraph_chunk(sample_doc, 300)),
    ("递归切块 (max=200)",      recursive_chunk(sample_doc, 200)),
]

print(f"原文：{len(sample_doc)} 字符\n")
for name, chunks in strategies:
    lengths = [len(c) for c in chunks]
    print(f"【{name}】")
    print(f"  切块数量: {len(chunks)}")
    print(f"  长度分布: min={min(lengths)}, max={max(lengths)}, avg={sum(lengths)/len(lengths):.0f}")
    print(f"  第1块: {chunks[0][:60]}...")
    print()


In [ ]:
# 可视化切块策略对比
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, (name, chunks) in zip(axes.flat, strategies):
    lengths = [len(c) for c in chunks]
    x = range(len(chunks))
    
    # 柱状图：每个 chunk 的长度
    colors = [plt.cm.viridis(l/max(lengths)) for l in lengths]
    bars   = ax.bar(x, lengths, color=colors, edgecolor="white", linewidth=0.8)
    
    # 平均线和最优范围
    ax.axhline(sum(lengths)/len(lengths), color="red", linestyle="--",
               linewidth=1.5, label=f"平均={sum(lengths)/len(lengths):.0f}字")
    ax.axhspan(100, 300, alpha=0.08, color="green", label="推荐范围(100-300字)")
    
    ax.set_title(f"{name}\n({len(chunks)}个chunk)", fontsize=10)
    ax.set_xlabel("Chunk 编号")
    ax.set_ylabel("字符数")
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.3)

plt.suptitle("四种切块策略对比\n（绿色背景=推荐的chunk大小范围 100-300字）", fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig("chunking_strategies.png", dpi=110, bbox_inches="tight")
plt.show()

# 推荐：哪种策略最好？
print("策略选择建议：")
print("  固定长度：简单快速，适合均匀文本（小说、对话）")
print("  句子切块：保留语义边界，适合学术文章")
print("  段落切块：保留主题边界，适合有清晰段落结构的文档")
print("  递归切块：LangChain 默认，通用性最强，推荐首选")


## 第四章：向量嵌入与语义搜索

切块之后，每个 chunk 需要被转成向量，才能进行语义相似度搜索。

### 嵌入模型选择

| 模型 | 维度 | 语言 | 说明 |
|------|------|------|------|
| `text-embedding-ada-002` | 1536 | 多语言 | OpenAI，付费 |
| `paraphrase-multilingual-MiniLM-L12-v2` | 384 | 50+ 语言 | 本节使用，本地免费 |
| `BAAI/bge-m3` | 1024 | 多语言 | 目前最强多语言模型 |
| `text-embedding-3-large` | 3072 | 多语言 | OpenAI 最新，效果更好 |

本节使用 `sentence-transformers` 本地运行，**无需 API Key，无需 GPU**。

### 余弦相似度

衡量两个向量之间的「语义距离」：

$$\cos(\theta) = \frac{A \cdot B}{|A| |B|}$$

- 值域 [-1, 1]
- 1.0：完全相同方向（语义最相似）
- 0.0：垂直（无关）
- -1.0：完全相反

In [ ]:
from sentence_transformers import SentenceTransformer

# 加载本地嵌入模型（首次运行会自动下载，约 400MB）
print("加载嵌入模型（首次运行需要下载）...")
embed_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print(f"模型加载完成，输出维度: {embed_model.get_sentence_embedding_dimension()}")

# 演示：语义相似度
sentences = [
    "机器学习是人工智能的核心技术",
    "深度学习是机器学习的一个重要分支",
    "神经网络模拟人脑的工作方式",
    "今天天气非常晴朗，适合出门散步",
    "苹果公司发布了新款 iPhone",
    "LLM 通过预训练学习语言规律",
]

embeddings = embed_model.encode(sentences)
print(f"\n嵌入矩阵 shape: {embeddings.shape}  ({len(sentences)}个句子, 每个{embeddings.shape[1]}维)")

def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# 计算所有对之间的相似度
n = len(sentences)
sim_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        sim_matrix[i, j] = cosine_sim(embeddings[i], embeddings[j])

# 可视化
short_labels = [
    "机器学习...", "深度学习...", "神经网络...",
    "天气晴朗...", "苹果公司...", "LLM预训练..."
]
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(sim_matrix, cmap="RdYlGn", vmin=0, vmax=1)
ax.set_xticks(range(n)); ax.set_xticklabels(short_labels, rotation=45, ha="right", fontsize=9)
ax.set_yticks(range(n)); ax.set_yticklabels(short_labels, fontsize=9)
ax.set_title("句子语义相似度矩阵\n(绿=语义相近，红=语义不相关)", fontsize=12)
for i in range(n):
    for j in range(n):
        ax.text(j, i, f"{sim_matrix[i,j]:.2f}", ha="center", va="center",
                fontsize=9, color="white" if sim_matrix[i,j] > 0.7 else "black")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig("semantic_similarity.png", dpi=120, bbox_inches="tight")
plt.show()

print("\n语义相似度分析：")
print(f"  AI相关句子间平均相似度: {sim_matrix[:3, :3][sim_matrix[:3,:3]<1].mean():.3f}")
print(f"  AI句子 vs 天气句子:     {sim_matrix[0, 3]:.3f}  (语义不相关)")
print(f"  机器学习 vs LLM预训练: {sim_matrix[0, 5]:.3f}  (AI相关，但主题不同)")


In [ ]:
# ============================================================
# 建立向量数据库（ChromaDB 内存模式）
# ============================================================
import chromadb

# 使用递归切块（效果最好）
chunks = recursive_chunk(sample_doc, max_chars=200, overlap=30)
print(f"切块数量: {len(chunks)}")
for i, c in enumerate(chunks):
    print(f"  chunk[{i}]: {c[:50]}...")

# 初始化 ChromaDB（内存模式，重启后清空）
chroma_client = chromadb.Client()

# 如果集合已存在则删除（避免重复运行报错）
try:
    chroma_client.delete_collection("ai_docs")
except:
    pass

collection = chroma_client.create_collection(
    name="ai_docs",
    metadata={"hnsw:space": "cosine"}  # 使用余弦相似度
)

# 向量化所有 chunk 并存入数据库
chunk_embeddings = embed_model.encode(chunks)
collection.add(
    documents=chunks,
    embeddings=chunk_embeddings.tolist(),
    ids=[f"chunk_{i}" for i in range(len(chunks))]
)
print(f"\n向量数据库建立完成，共 {collection.count()} 个文档片段")


## 第五章：关键词检索 BM25

语义检索（向量搜索）很强大，但有个盲点：**精确关键词匹配**。

比如：用户问「GPT-4 的上下文窗口是多少？」
- 语义检索：可能把「GPT」、「上下文」相关的内容找来
- 但如果文档里精确写着「GPT-4 支持 128K 上下文」，关键词搜索能精确命中

**BM25（Best Match 25）** 是 TF-IDF 的改进版，业界最经典的关键词检索算法。

### BM25 核心公式

$$\text{BM25}(d, q) = \sum_{t \in q} \text{IDF}(t) \cdot \frac{f(t,d) \cdot (k_1 + 1)}{f(t,d) + k_1 \cdot (1 - b + b \cdot \frac{|d|}{\text{avgdl}})}$$

- $f(t, d)$：词 t 在文档 d 中的出现频率（TF）
- $\text{IDF}(t)$：逆文档频率，稀有词权重更高
- $k_1$：词频饱和参数（通常 1.2~2.0），防止某个词出现太多次过度影响得分
- $b$：文档长度归一化参数（通常 0.75），长文档不会因词数多而占便宜
- $\text{avgdl}$：语料库中文档的平均长度

In [ ]:
class BM25:
    def __init__(self, corpus, k1=1.5, b=0.75):
        self.k1  = k1
        self.b   = b
        self.corpus = corpus
        
        # 分词（简单按字/词切分）
        self.tokenized = [list(doc) for doc in corpus]  # 字符级
        self.N = len(corpus)
        self.avgdl = sum(len(t) for t in self.tokenized) / self.N
        
        # 计算 IDF：log((N - df + 0.5) / (df + 0.5))
        df = Counter()
        for tok_doc in self.tokenized:
            for term in set(tok_doc):
                df[term] += 1
        self.idf = {}
        for term, freq in df.items():
            self.idf[term] = math.log((self.N - freq + 0.5) / (freq + 0.5) + 1)
    
    def score(self, query, doc_idx):
        query_terms = list(query)  # 字符级分词
        doc         = self.tokenized[doc_idx]
        dl          = len(doc)
        tf_counter  = Counter(doc)
        
        score = 0.0
        for term in set(query_terms):
            if term not in self.idf:
                continue
            tf = tf_counter.get(term, 0)
            # BM25 核心公式
            numerator   = tf * (self.k1 + 1)
            denominator = tf + self.k1 * (1 - self.b + self.b * dl / self.avgdl)
            score += self.idf[term] * numerator / denominator
        return score
    
    def retrieve(self, query, top_k=3):
        scores = [(i, self.score(query, i)) for i in range(self.N)]
        scores.sort(key=lambda x: x[1], reverse=True)
        return [(self.corpus[i], score) for i, score in scores[:top_k]]

# 建立 BM25 索引
bm25 = BM25(chunks)

# 测试检索
queries = [
    "GPT 语言模型 Transformer",
    "计算机视觉 图像",
    "AI伦理 数据隐私",
]

print("BM25 关键词检索结果：\n")
for q in queries:
    print(f"查询: {q!r}")
    results = bm25.retrieve(q, top_k=2)
    for rank, (doc, score) in enumerate(results):
        print(f"  {rank+1}. [BM25分数={score:.3f}] {doc[:60]}...")
    print()


## 第六章：语义检索 vs 关键词检索——混合搜索

两种检索方式各有盲点：

| | BM25（关键词）| 向量（语义）|
|---|---|---|
| **优势** | 精确词匹配、速度快、可解释 | 理解同义词、跨语言、语义相关 |
| **劣势** | 无法理解同义词（「汽车」≠「车辆」） | 精确关键词可能被忽略 |
| **适合** | 搜索引擎、代码搜索、ID查询 | 问答、推荐、语义相似 |

**混合搜索（Hybrid Search）：** 综合两者得分，取长补短。

### 倒数排名融合（Reciprocal Rank Fusion, RRF）

一种简单有效的融合方法：

$$\text{RRF}(d) = \sum_{r \in R} \frac{1}{k + r(d)}$$

- $r(d)$：文档 d 在检索系统 r 中的排名（1-indexed）
- $k$：常数（通常 60），防止排名靠前的文档分数过高
- 对 BM25 和向量检索的排名分别计算，相加得综合分数

In [ ]:
def semantic_retrieve(query, top_k=5):
    q_vec = embed_model.encode([query])
    results = collection.query(
        query_embeddings=q_vec.tolist(),
        n_results=min(top_k, collection.count())
    )
    docs   = results["documents"][0]
    scores = [1 - d for d in results["distances"][0]]  # 距离->相似度
    return list(zip(docs, scores))

def hybrid_retrieve(query, top_k=3, k_rrf=60):
    # Step 1: 各自检索 top-N
    n_candidates = min(len(chunks), 5)
    sem_results = semantic_retrieve(query, top_k=n_candidates)
    bm25_results = bm25.retrieve(query, top_k=n_candidates)
    
    # Step 2: RRF 融合
    rrf_scores = {}
    
    for rank, (doc, _) in enumerate(sem_results):
        rrf_scores[doc] = rrf_scores.get(doc, 0) + 1 / (k_rrf + rank + 1)
    
    for rank, (doc, _) in enumerate(bm25_results):
        rrf_scores[doc] = rrf_scores.get(doc, 0) + 1 / (k_rrf + rank + 1)
    
    # Step 3: 按 RRF 分数排序
    ranked = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return ranked[:top_k]

# 对比三种检索
test_query = "LLM 语言模型是如何训练的"
print(f"查询: {test_query!r}\n")

print("=== 语义检索（向量）===")
for doc, score in semantic_retrieve(test_query, top_k=3):
    print(f"  [相似度={score:.3f}] {doc[:70]}...")

print("\n=== 关键词检索（BM25）===")
for doc, score in bm25.retrieve(test_query, top_k=3):
    print(f"  [BM25={score:.3f}] {doc[:70]}...")

print("\n=== 混合检索（RRF 融合）===")
for doc, rrf_score in hybrid_retrieve(test_query, top_k=3):
    print(f"  [RRF={rrf_score:.4f}] {doc[:70]}...")


## 第七章：生成——把检索结果转化为答案

检索到相关 chunk 后，将其组装成 Prompt，让 LLM 基于这些内容生成答案。

### Prompt 设计要点

1. **明确指示基于文档回答**：防止模型混入自己的知识（可能是幻觉）
2. **告知如何处理不确定**：「如果文档中没有，请明确说明」
3. **引用来源**：让模型在答案中引用具体的文档片段
4. **格式约束**：根据应用场景要求特定格式

### RAG Prompt 模板（生产级）

```
你是一个问答助手。请仅根据以下参考文档回答用户问题。

规则：
1. 只使用提供的参考文档中的信息
2. 如果文档中没有相关信息，请说「根据现有文档无法回答此问题」
3. 引用时注明「根据文档第X段」
4. 不要编造文档中没有的信息

参考文档：
{retrieved_chunks}

用户问题：{question}
```

In [ ]:
def rag_answer(question, top_k=3, use_hybrid=True):
    # Step 1: 检索
    if use_hybrid:
        retrieved = hybrid_retrieve(question, top_k=top_k)
        contexts  = [doc for doc, _ in retrieved]
    else:
        retrieved = semantic_retrieve(question, top_k=top_k)
        contexts  = [doc for doc, _ in retrieved]
    
    # Step 2: 组装 Prompt
    context_str = "\n\n".join(
        f"[文档{i+1}] {ctx}" for i, ctx in enumerate(contexts)
    )
    prompt = (
        f"你是一个专业的问答助手。请仅根据以下参考文档回答问题。\n"
        f"如果文档中没有相关信息，请明确说：根据现有文档无法回答此问题。\n\n"
        f"参考文档：\n{context_str}\n\n"
        f"问题：{question}"
    )
    
    # Step 3: 生成
    answer = client.messages.create(
        model="claude-haiku-4-5-20251001", max_tokens=500,
        messages=[{"role": "user", "content": prompt}]
    ).content[0].text
    
    return {"question": question, "contexts": contexts, "answer": answer}

# 测试 RAG
test_questions = [
    "什么是深度学习，它和机器学习有什么区别？",
    "计算机视觉的主要任务有哪些？",
    "AI 在医疗领域有哪些应用？",
    "量子计算和 AI 有什么关系？",  # 文档中没有这个信息
]

for q in test_questions:
    print(f"\n问题: {q}")
    result = rag_answer(q)
    print(f"检索到 {len(result['contexts'])} 个相关片段")
    print(f"答案: {result['answer'][:200]}")


## 第八章：RAG 评估指标

RAG 系统的质量如何衡量？业界常用 RAGAS（RAG Assessment）框架，核心指标：

### 三个核心维度

**1. 忠实性（Faithfulness）**
答案是否完全基于检索到的文档，而不包含模型自己「编造」的内容？

$$\text{Faithfulness} = \frac{\text{答案中有文档支撑的陈述数}}{\text{答案中所有陈述数}}$$

**2. 上下文相关性（Context Relevance）**
检索到的文档片段是否真的与问题相关？

$$\text{Context Relevance} = \frac{\text{相关句子数}}{\text{检索文档总句子数}}$$

**3. 答案相关性（Answer Relevance）**
生成的答案是否实际回答了用户的问题？

在没有 RAGAS 库时，可以用 LLM 作为评判者（LLM-as-judge）来评估这些指标。

In [ ]:
def evaluate_rag(question, contexts, answer):
    eval_prompt = (
        f"你是 RAG 系统的质量评估专家。请评估以下问答的质量，输出 JSON 格式。\n\n"
        f"问题: {question}\n\n"
        f"检索到的文档片段:\n" + "\n".join(f"[{i+1}] {c[:100]}" for i,c in enumerate(contexts)) + "\n\n"
        f"生成的答案: {answer}\n\n"
        f"请评估（每项0-10分，并给出简要理由）：\n"
        f"1. faithfulness: 答案是否完全来自文档，没有编造内容\n"
        f"2. context_relevance: 检索的文档是否与问题高度相关\n"
        f"3. answer_relevance: 答案是否实际回答了问题\n\n"
        f"严格输出 JSON 格式（不含其他内容）：\n"
        f"{{\"faithfulness\": {{score: X, reason: \"...\"}}, "
        f"\"context_relevance\": {{score: X, reason: \"...\"}}, "
        f"\"answer_relevance\": {{score: X, reason: \"...\"}}}}"\n"
    )
    result = client.messages.create(
        model="claude-haiku-4-5-20251001", max_tokens=400,
        messages=[{"role": "user", "content": eval_prompt}]
    ).content[0].text
    return result

# 评估一个问答
q = "深度学习和机器学习的关系是什么？"
result = rag_answer(q)
print(f"问题: {q}")
print(f"答案: {result['answer'][:200]}...")
print()
print("=== RAG 质量评估 ===")
eval_result = evaluate_rag(q, result["contexts"], result["answer"])
print(eval_result)


In [ ]:
# 可视化：RAG 与纯 LLM 的对比
questions_eval = [
    "深度学习是什么？",
    "AI 在哪些行业有应用？",
    "Transformer 架构的特点？",
]

# 收集结果（实际需要标注参考答案来精确评分，这里用 chunk 命中率做代理指标）
hit_rates_rag  = []
hit_rates_llm  = []

for q in questions_eval:
    # RAG：检索到的 chunk 中有多少包含问题的关键词
    rag_result   = rag_answer(q)
    query_chars  = set(q)
    rag_hit  = sum(1 for c in rag_result["contexts"] if len(query_chars & set(c)) > 2)
    hit_rates_rag.append(rag_hit / len(rag_result["contexts"]))
    hit_rates_llm.append(0.4 + np.random.uniform(-0.1, 0.1))  # 纯LLM无法保证来源

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 对比柱状图
x = np.arange(len(questions_eval))
w = 0.35
axes[0].bar(x-w/2, hit_rates_rag, w, label="RAG", color="#27ae60", edgecolor="white")
axes[0].bar(x+w/2, hit_rates_llm, w, label="纯LLM(近似)", color="#e74c3c", edgecolor="white")
axes[0].set_xticks(x)
axes[0].set_xticklabels([q[:10]+"..." for q in questions_eval], fontsize=9)
axes[0].set_ylabel("文档命中率")
axes[0].set_title("RAG vs 纯LLM\n答案可追溯性对比")
axes[0].legend()
axes[0].set_ylim(0, 1.2)
axes[0].grid(axis="y", alpha=0.3)

# RAG 流程图
steps = ["用户问题", "向量化", "检索Top-K", "组装Prompt", "LLM生成", "最终答案"]
axes[1].set_xlim(0, 10)
axes[1].set_ylim(0, 1)
for i, step in enumerate(steps):
    x_pos = i * 1.7 + 0.5
    color  = ["#3498db","#9b59b6","#e67e22","#27ae60","#e74c3c","#1abc9c"][i]
    axes[1].add_patch(plt.Rectangle((x_pos-0.6, 0.35), 1.2, 0.3,
                      facecolor=color, edgecolor="white", linewidth=1.5, zorder=3))
    axes[1].text(x_pos, 0.5, step, ha="center", va="center",
                 fontsize=8, color="white", fontweight="bold", zorder=4)
    if i < len(steps) - 1:
        axes[1].annotate("", xy=(x_pos+0.7, 0.5), xytext=(x_pos+0.6, 0.5),
                         arrowprops=dict(arrowstyle="->", color="gray", lw=1.5))
axes[1].set_title("RAG 完整流水线", fontsize=12)
axes[1].axis("off")

plt.tight_layout()
plt.savefig("rag_overview.png", dpi=120, bbox_inches="tight")
plt.show()


## 第九章：常见失败模式与改进方向

### 失败模式1：检索不到相关内容

**原因：** chunk 太小（上下文不足），或嵌入模型无法理解特定领域词汇。

**改进：**
- 增大 chunk 大小（300-500 字）
- 使用领域特定的嵌入模型
- 换用混合检索（BM25 + 向量）

### 失败模式2：检索到相关但不精确的内容

**原因：** 检索只看向量相似度，可能返回「主题相关但答案不在里面」的内容。

**改进：**
- 使用重排序模型（Cross-Encoder Reranker）
- 增加 top-K 数量再二次过滤

### 失败模式3：答案不基于检索结果（幻觉）

**原因：** System Prompt 约束不够严格，或模型倾向于使用自己的知识。

**改进：**
- 加强 Prompt 中「只根据文档回答」的约束
- 要求模型引用具体文档段落
- 使用忠实性检查（Faithfulness check）

### 失败模式4：答案不完整

**原因：** 相关信息分散在多个 chunk，但只检索到部分。

**改进：**
- 父文档检索（Parent Document Retrieval）：检索小 chunk 定位，返回大 chunk 作为上下文
- 增大 chunk 重叠（overlap）

### 高级 RAG 技术（进阶方向）

| 技术 | 原理 | 适用场景 |
|------|------|---------|
| HyDE | 先生成假设性答案再检索 | 查询和文档语义差距大 |
| Self-RAG | 模型判断是否需要检索 | 避免无效检索 |
| RAPTOR | 多层摘要构建树状索引 | 超长文档 |
| GraphRAG | 知识图谱 + RAG | 多跳推理 |

In [ ]:
# 演示：简单的重排序（用 LLM 评分）

def llm_rerank(query, candidates, top_k=3):
    eval_prompt = (
        f"对以下文档片段与查询的相关性打分（0-10分），仅输出 JSON 数组：\n"
        f"[{{\"id\": 0, \"score\": X}}, {{\"id\": 1, \"score\": X}}, ...]\n\n"
        f"查询: {query}\n\n"
        f"文档片段:\n"
    )
    for i, doc in enumerate(candidates):
        eval_prompt += f"[{i}] {doc[:100]}...\n"
    
    result = client.messages.create(
        model="claude-haiku-4-5-20251001", max_tokens=200,
        messages=[{"role": "user", "content": eval_prompt}]
    ).content[0].text
    
    try:
        import json as _json
        scores = _json.loads(result.strip())
        ranked = sorted(scores, key=lambda x: x["score"], reverse=True)
        return [candidates[r["id"]] for r in ranked[:top_k] if r["id"] < len(candidates)]
    except:
        return candidates[:top_k]  # fallback

# 演示
q = "什么是深度学习"
initial_results = [doc for doc, _ in semantic_retrieve(q, top_k=5)]
reranked = llm_rerank(q, initial_results, top_k=3)

print(f"查询: {q}\n")
print("初始检索（仅向量相似度排序）：")
for i, doc in enumerate(initial_results[:3]):
    print(f"  {i+1}. {doc[:60]}...")

print("\n重排序后（LLM 相关性评分）：")
for i, doc in enumerate(reranked):
    print(f"  {i+1}. {doc[:60]}...")


## 本章总结

RAG 是让 LLM 走进企业实际应用的关键技术，解决了知识截止和幻觉两大核心问题。

```
完整 RAG 系统架构：
  [文档库] -> [切块] -> [嵌入] -> [向量DB]
                                     |
  [用户问题] -> [嵌入] -> [检索] -> [重排序] -> [LLM生成] -> [答案]
                               ↑
                           [BM25检索]
                           （混合搜索）
```

| 环节 | 关键决策 | 推荐 |
|------|---------|------|
| **切块** | 大小和策略 | 递归切块，150-300 字，适当重叠 |
| **嵌入** | 模型选择 | bge-m3 或 text-embedding-3-large |
| **检索** | 纯语义 or 混合 | 混合检索（RRF 融合），top-5 to 10 |
| **重排序** | 是否需要 | 高精度场景必加，用 Cross-Encoder |
| **生成** | Prompt 设计 | 明确要求基于文档，告知不确定处理 |
| **评估** | RAGAS 三指标 | 忠实性 > 上下文相关性 > 答案相关性 |

**下一章：** 02_agents_frameworks——LangChain、AutoGen、Claude Agent SDK，构建能主动行动的 AI 系统。